In [ ]:
#

import os
import re
import json
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

# ================= CONFIG =================
LABEL_CSV = "../LiverFibrosisCohort/LiverFibrosisCohort_MasterTable_Proccessed_519.csv"
RAD_ROOT = "/mnt/12T_Ironwolf/LiverData/Radiomics"
DEEP_ROOT = "../LiverFibrosisCohort/deep_medsiglip_v2"

ID_COLUMN = "Image ID"
LABEL_COLUMN = "Fibrosis_Label"
SITE_COLUMN = "SITE"

DEV_SITES = ["CCHMC", "WISC","MICH"]
EXTERNAL_SITES = ["NYU"]


# DEV_SITES = ["CCHMC", "MICH","NYU"]
# EXTERNAL_SITES = ["WISC"]

ALL_SITES = DEV_SITES + EXTERNAL_SITES

IMAGE_MODALITIES = ["ADC", "T1", "T2"]
ALL_MODALITIES = ["ADC", "T1", "T2", "MRE", "CLINICAL"]

# Keep the same branch widths as the current baseline for a cleaner architecture comparison.
BRANCH_CONFIG = {
    "ADC_RAD":      {"hidden": 16},
    "ADC_DEEP":     {"hidden": 64},
    "T1_RAD":       {"hidden": 8},
    "T1_DEEP":      {"hidden": 128},
    "T2_RAD":       {"hidden": 8},
    "T2_DEEP":      {"hidden": 128},
    "MRE":          {"hidden": 16},
    "CLINICAL":     {"hidden": 32},
}

child_cutoff=18
# New architecture settings.
COMMON_EMBED_DIM = 64
FUSION_HIDDEN_DIMS = [256, 128]
ENCODER_DROPOUT = 0.20
FUSION_DROPOUT = 0.1
MODALITY_DROPOUT = 0.1
AUX_LOSS_WEIGHT = 0.03

SEED = 2
N_SPLITS = 5
EPOCHS = 100
LR = 5e-4
WEIGHT_DECAY = 3e-4
BATCH_SIZE = 16
FIXED_THRESHOLD = 0.5
CV_INNER_VAL_FRAC = 0.15
FINAL_RETRAIN_VAL_FRAC = 0.20
EARLY_STOPPING_PATIENCE = 20
MIN_EPOCHS = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = "../Results/gated_hierarchical_multimodal_fusion"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MERGE_AUDIT_DIR = os.path.join(OUTPUT_DIR, "merge_audit")
os.makedirs(MERGE_AUDIT_DIR, exist_ok=True)

# ================= TABULAR FEATURE SETS =================
COMMON_NUMERIC_COLS = ["AGE_years", "BMI"]
COMMON_CATEGORICAL_COLS = ["SEX", "RACE", "ETHNICITY"]

MRE_NUMERIC_COLS = ["Stiffness(kPa)", "PDFF"]
MRE_CATEGORICAL_COLS = []

CLINICAL_NUMERIC_COLS = [
    "TBIL (mg/dL)",
    "ALT (U/L)",
    "A1C (%)",
    "GLUC (mg/dL)",
    "DBIL (mg/dL)",
    "ALB (g/dL)",
    "PLT (K/uL)",
    "AST (U/L)",
    "ALK (U/L)",
    "HCT (%)",
] + COMMON_NUMERIC_COLS

CLINICAL_HISTORY_COLS = [
    "Alcohol_Liver_Disease",
    "Fatty_Liver_Disease(NAFLD_NASH)",
    "Primary_Sclerosing_Cholangitis(PSC)",
    "Liver_Transplantation",
    "Primary_Biliary_Cirrhosis(PBC)",
    "Hepatitis_B_C",
    "Autoimmune_Hepatitis(AIH)",
    "Diabetes_Type2",
]

CLINICAL_CATEGORICAL_COLS = COMMON_CATEGORICAL_COLS + CLINICAL_HISTORY_COLS


# ================= REPRODUCIBILITY =================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ================= HELPERS =================
def normalize_id(pid, strip_wisc_date=False):
    pid = str(pid).strip()

    m = re.match(r"^PT-(\d{3}-\d{4})-", pid)
    if m:
        return m.group(1)

    m = re.match(r"^PT-(M\d+)-", pid)
    if m:
        return m.group(1)

    if strip_wisc_date:
        m = re.match(r"^(GJ[A-Z0-9]+)_\d{8}$", pid)
        if m:
            return m.group(1)

    return pid


def clean_site(x):
    s = str(x).strip().upper()

    if s in ["CCHMC1", "CCHMC2"] or s.startswith("CCHMC"):
        return "CCHMC"

    if s in ["UW1", "UW2"] or s.startswith("UW") or "WISC" in s:
        return "WISC"

    if s == "NYU" or s.startswith("NYU"):
        return "NYU"

    if (
        s == "UM"
        or s.startswith("UM")
        or s == "MICH"
        or s.startswith("MICH")
        or "MICHIGAN" in s
    ):
        return "MICH"

    return "UNKNOWN"


def get_site_from_id(pid):
    pid = str(pid).strip().upper()

    if re.match(r"^\d{3}-\d{4}$", pid):
        return "CCHMC"
    elif pid.startswith("GJ"):
        return "WISC"
    elif pid.startswith("C"):
        return "MICH"
    elif re.match(r"^\d+$", pid):
        return "NYU"
    else:
        return "UNKNOWN"


def binarize_fibrosis(x):
    if pd.isna(x):
        return np.nan

    s = str(x)
    m = re.search(r"\d+", s)
    if m is None:
        return np.nan

    val = int(m.group())
    if val in [0, 1]:
        return 0
    if val in [2, 3, 4]:
        return 1
    return np.nan


def mean_std(series):
    vals = pd.Series(series).dropna().values
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), 0.0
    return float(np.mean(vals)), float(np.std(vals, ddof=1))


def save_csv(df, path):
    df.to_csv(path, index=False)


# ================= SPLIT STRATIFICATION HELPERS =================
def make_site_label_strata(df, site_col="site", label_col="label"):
    """
    Joint stratification key so CV folds preserve both site distribution
    and fibrosis class distribution as much as possible.

    Example strata: CCHMC_y0, CCHMC_y1, WISC_y0, WISC_y1, ...
    """
    if site_col not in df.columns:
        raise ValueError(f"Missing site column for stratification: {site_col}")
    if label_col not in df.columns:
        raise ValueError(f"Missing label column for stratification: {label_col}")

    site = df[site_col].fillna("UNKNOWN").astype(str)
    label = df[label_col].astype(int).astype(str)
    return site + "_y" + label


def print_site_label_counts(df, context):
    counts = (
        df.groupby(["site", "label"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["site", "label"])
    )
    print(f"\n[{context}] site x label counts:")
    print(counts.to_string(index=False))
    return counts


def choose_cv_strata(df, n_splits, context="dev_cv"):
    """
    Prefer joint site+label stratification.
    If any site-label cell has fewer than n_splits samples, perfect joint
    stratification is impossible. In that case, fall back to label-only
    stratification instead of silently producing unstable folds.
    """
    site_label = make_site_label_strata(df)
    counts = site_label.value_counts().sort_index()
    bad = counts[counts < n_splits]

    print_site_label_counts(df, context)

    if bad.empty:
        print(f"[{context}] using site+label stratification for {n_splits}-fold CV")
        return site_label, "site_label"

    print(
        f"[{context}] WARNING: cannot guarantee site+label stratification with "
        f"n_splits={n_splits}. These strata have too few samples: "
        f"{bad.to_dict()}"
    )

    label_counts = df["label"].astype(int).value_counts().sort_index()
    if (label_counts < n_splits).any():
        raise ValueError(
            f"Cannot run {n_splits}-fold StratifiedKFold: label counts are "
            f"{label_counts.to_dict()}"
        )

    print(f"[{context}] falling back to label-only stratification")
    return df["label"].astype(int), "label_only_fallback"


def choose_train_test_strata(df, context="train_val_split"):
    """
    Prefer site+label stratification for train/validation splits.
    train_test_split requires every stratum to have at least 2 samples.
    """
    site_label = make_site_label_strata(df)
    counts = site_label.value_counts().sort_index()

    if len(counts) > 1 and counts.min() >= 2:
        print(f"[{context}] using site+label stratification")
        return site_label

    print(
        f"[{context}] WARNING: cannot use site+label stratification because "
        f"some strata have <2 samples: {counts[counts < 2].to_dict()}"
    )

    label = df["label"].astype(int)
    label_counts = label.value_counts().sort_index()
    if len(label_counts) > 1 and label_counts.min() >= 2:
        print(f"[{context}] falling back to label-only stratification")
        return label

    print(f"[{context}] falling back to unstratified split")
    return None


def add_fold_site_label_counts(row, fold_idx, split_name, split_df):
    """Add compact per-site/per-label counts to a fold result row."""
    row[f"{split_name}_n"] = int(len(split_df))
    for site in sorted(split_df["site"].dropna().astype(str).unique()):
        site_mask = split_df["site"].astype(str) == site
        row[f"{split_name}_{site}_n"] = int(site_mask.sum())
        row[f"{split_name}_{site}_neg"] = int((site_mask & (split_df["label"].astype(int) == 0)).sum())
        row[f"{split_name}_{site}_pos"] = int((site_mask & (split_df["label"].astype(int) == 1)).sum())
    return row


def audit_normalization_collisions(df, name, raw_id_col="raw_id", id_col="id", audit_dir=MERGE_AUDIT_DIR):
    if raw_id_col not in df.columns or id_col not in df.columns:
        return

    collision_counts = (
        df.groupby(id_col)[raw_id_col]
        .nunique(dropna=False)
        .reset_index(name="n_raw_ids")
    )
    collision_counts = collision_counts[collision_counts["n_raw_ids"] > 1].copy()

    if collision_counts.empty:
        print(f"[{name}] normalization collisions: 0")
        return

    raw_lists = (
        df.groupby(id_col)[raw_id_col]
        .apply(lambda s: " | ".join(sorted(s.astype(str).unique().tolist())[:50]))
        .reset_index(name="raw_ids")
    )
    out = collision_counts.merge(raw_lists, on=id_col, how="left")
    save_csv(out, os.path.join(audit_dir, f"{name}_normalization_collisions.csv"))
    print(f"[{name}] normalization collisions: {len(out)} ids")


def audit_site_consistency(df, name, id_col="id", site_col="site", audit_dir=MERGE_AUDIT_DIR):
    if id_col not in df.columns or site_col not in df.columns:
        return

    tmp = df[[id_col, site_col]].copy()
    tmp["site_from_id"] = tmp[id_col].apply(get_site_from_id)

    mismatches = tmp[
        (tmp[site_col] != "UNKNOWN")
        & (tmp["site_from_id"] != "UNKNOWN")
        & (tmp[site_col] != tmp["site_from_id"])
    ].copy()

    print(f"[{name}] site mismatches vs ID pattern: {len(mismatches)}")

    if not mismatches.empty:
        save_csv(mismatches, os.path.join(audit_dir, f"{name}_site_mismatches.csv"))


def audit_and_resolve_duplicates(df, name, id_col="id", raw_id_col=None, audit_dir=MERGE_AUDIT_DIR):
    dup_df = df[df.duplicated(subset=[id_col], keep=False)].copy()

    print(
        f"[{name}] rows={len(df)}, unique_ids={df[id_col].nunique()}, "
        f"duplicate_ids={dup_df[id_col].nunique() if len(dup_df) else 0}"
    )

    if dup_df.empty:
        return df.reset_index(drop=True)

    summary_rows = []
    for pid, g in dup_df.groupby(id_col, sort=False):
        payload = g.drop(columns=[c for c in [id_col] if c in g.columns]).copy()
        n_unique_payload_rows = len(payload.astype(str).drop_duplicates())

        row = {
            "id": pid,
            "n_rows": len(g),
            "n_unique_payload_rows": n_unique_payload_rows,
        }
        if raw_id_col is not None and raw_id_col in g.columns:
            row["raw_ids"] = " | ".join(sorted(g[raw_id_col].astype(str).unique().tolist())[:50])

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["n_unique_payload_rows", "n_rows"], ascending=False
    )

    save_csv(summary_df, os.path.join(audit_dir, f"{name}_duplicate_id_summary.csv"))
    save_csv(dup_df, os.path.join(audit_dir, f"{name}_duplicate_id_rows.csv"))

    conflict_df = summary_df[summary_df["n_unique_payload_rows"] > 1].copy()
    if not conflict_df.empty:
        raise ValueError(
            f"{name}: conflicting duplicate rows after ID normalization. "
            f"See {os.path.join(audit_dir, f'{name}_duplicate_id_summary.csv')}"
        )

    print(f"[{name}] exact duplicate IDs found, keeping first occurrence")
    return df.drop_duplicates(subset=[id_col], keep="first").reset_index(drop=True)


def audit_key_overlap(left_df, right_df, left_name, right_name, key="id", audit_dir=MERGE_AUDIT_DIR):
    left_keys = left_df[[key]].drop_duplicates().copy()
    right_keys = right_df[[key]].drop_duplicates().copy()

    overlap = left_keys.merge(right_keys, on=key, how="outer", indicator=True)

    left_only = overlap[overlap["_merge"] == "left_only"].copy()
    right_only = overlap[overlap["_merge"] == "right_only"].copy()
    both = overlap[overlap["_merge"] == "both"].copy()

    base = f"{left_name}__vs__{right_name}"
    summary = pd.DataFrame([{
        "left_name": left_name,
        "right_name": right_name,
        "left_rows": len(left_df),
        "right_rows": len(right_df),
        "left_unique_ids": left_df[key].nunique(),
        "right_unique_ids": right_df[key].nunique(),
        "matched_ids": len(both),
        "left_only_ids": len(left_only),
        "right_only_ids": len(right_only),
    }])

    save_csv(summary, os.path.join(audit_dir, f"{base}_key_overlap_summary.csv"))
    if not left_only.empty:
        save_csv(left_only, os.path.join(audit_dir, f"{base}_left_only_ids.csv"))
    if not right_only.empty:
        save_csv(right_only, os.path.join(audit_dir, f"{base}_right_only_ids.csv"))

    print(summary.to_string(index=False))


def audited_left_merge(left_df, right_df, left_name, right_name, key="id", audit_dir=MERGE_AUDIT_DIR):
    audit_key_overlap(left_df, right_df, left_name, right_name, key=key, audit_dir=audit_dir)

    merged = left_df.merge(
        right_df,
        on=key,
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    merge_counts = merged["_merge"].value_counts(dropna=False).to_dict()
    print(
        f"[MERGE] {left_name} <- {right_name}: "
        f"before={len(left_df)}, after={len(merged)}, "
        f"matched={merge_counts.get('both', 0)}, "
        f"left_only={merge_counts.get('left_only', 0)}"
    )

    if len(merged) != len(left_df):
        raise ValueError(
            f"Row count changed during left merge: {left_name} <- {right_name}. "
            f"This usually means duplicate IDs slipped through."
        )

    merged = merged.drop(columns=["_merge"])
    return merged


def add_presence_marker(df, present_name, cols):
    cols = [c for c in cols if c in df.columns]
    if len(cols) == 0:
        df[present_name] = 0
        return df

    present = pd.Series(False, index=df.index)
    for c in cols:
        present = present | df[c].notna()

    df[present_name] = present.astype(int)
    return df


def summarize_branch_presence(merged_df, audit_dir=MERGE_AUDIT_DIR):
    present_cols = [c for c in merged_df.columns if c.startswith("PRESENT_")]
    if not present_cols:
        return

    rows = []
    for col in present_cols:
        tmp = (
            merged_df.assign(_present=merged_df[col].fillna(0).astype(float))
            .groupby("site", dropna=False)["_present"]
            .mean()
            .reset_index(name="present_rate")
        )
        tmp["branch"] = col.replace("PRESENT_", "")
        rows.append(tmp)

    out = pd.concat(rows, ignore_index=True)
    save_csv(out, os.path.join(audit_dir, "branch_presence_by_site.csv"))

    wide = out.pivot(index="site", columns="branch", values="present_rate").fillna(0).round(3)
    print("\nBranch presence by site:")
    print(wide)

    id_cols = ["id", "site", "label"] + present_cols
    save_csv(merged_df[id_cols], os.path.join(audit_dir, "merged_presence_matrix.csv"))


# ================= DATA LOADING =================
def load_master_table(audit_dir=MERGE_AUDIT_DIR):
    df = pd.read_csv(LABEL_CSV).copy()

    df["raw_id"] = df[ID_COLUMN].astype(str)
    df["id"] = df["raw_id"].apply(normalize_id)
    audit_normalization_collisions(df, "master_raw", raw_id_col="raw_id", id_col="id", audit_dir=audit_dir)

    if "AGE_years" not in df.columns and "AGE_days" in df.columns:
        df["AGE_years"] = pd.to_numeric(df["AGE_days"], errors="coerce") / 365.25

    df["label"] = df[LABEL_COLUMN].apply(binarize_fibrosis)
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    if SITE_COLUMN in df.columns:
        df["site"] = df[SITE_COLUMN].apply(clean_site)
    else:
        df["site"] = df["id"].apply(get_site_from_id)

    audit_site_consistency(df, "master_before_site_filter", id_col="id", site_col="site", audit_dir=audit_dir)

    df = df[df["site"].isin(ALL_SITES)].copy()
    df = audit_and_resolve_duplicates(
        df,
        name="master_filtered",
        id_col="id",
        raw_id_col="raw_id",
        audit_dir=audit_dir,
    )

    save_csv(
        df[["raw_id", "id", "site", "label"]],
        os.path.join(audit_dir, "master_id_map.csv"),
    )

    print(
        f"[master_filtered] final rows={len(df)}, unique_ids={df['id'].nunique()}, "
        f"site_counts={df['site'].value_counts(dropna=False).sort_index().to_dict()}"
    )

    return df.reset_index(drop=True)


def load_radiomics(modality, audit_dir=MERGE_AUDIT_DIR):
    liver_path = os.path.join(RAD_ROOT, f"{modality}_liver_features.csv")
    spleen_path = os.path.join(RAD_ROOT, f"{modality}_spleen_features.csv")

    if not os.path.exists(liver_path):
        raise FileNotFoundError(f"Missing file: {liver_path}")
    if not os.path.exists(spleen_path):
        raise FileNotFoundError(f"Missing file: {spleen_path}")

    liver_df = pd.read_csv(liver_path).copy()
    spleen_df = pd.read_csv(spleen_path).copy()

    liver_df["raw_id"] = liver_df["ID"].astype(str)
    spleen_df["raw_id"] = spleen_df["ID"].astype(str)

    liver_df["id"] = liver_df["raw_id"].apply(normalize_id)
    spleen_df["id"] = spleen_df["raw_id"].apply(normalize_id)

    audit_normalization_collisions(liver_df, f"{modality}_liver_raw", raw_id_col="raw_id", id_col="id", audit_dir=audit_dir)
    audit_normalization_collisions(spleen_df, f"{modality}_spleen_raw", raw_id_col="raw_id", id_col="id", audit_dir=audit_dir)

    liver_df = liver_df.drop(columns=["ID"])
    spleen_df = spleen_df.drop(columns=["ID"])

    liver_df = audit_and_resolve_duplicates(
        liver_df,
        name=f"{modality}_liver",
        id_col="id",
        raw_id_col="raw_id",
        audit_dir=audit_dir,
    )
    spleen_df = audit_and_resolve_duplicates(
        spleen_df,
        name=f"{modality}_spleen",
        id_col="id",
        raw_id_col="raw_id",
        audit_dir=audit_dir,
    )

    audit_key_overlap(
        liver_df,
        spleen_df,
        left_name=f"{modality}_liver",
        right_name=f"{modality}_spleen",
        key="id",
        audit_dir=audit_dir,
    )

    rad_df = liver_df.merge(
        spleen_df,
        on="id",
        how="inner",
        validate="one_to_one",
        suffixes=("_liver", "_spleen"),
    )

    rad_df[f"PRESENT_{modality}_RAD"] = 1

    drop_cols = [c for c in ["raw_id_liver", "raw_id_spleen"] if c in rad_df.columns]
    if drop_cols:
        rad_df = rad_df.drop(columns=drop_cols)

    rename_map = {}
    for c in rad_df.columns:
        if c not in ["id", f"PRESENT_{modality}_RAD"]:
            rename_map[c] = f"{modality}_RAD_{c}"

    rad_df = rad_df.rename(columns=rename_map)

    print(
        f"[{modality}_RAD] rows after liver/spleen inner merge={len(rad_df)}, "
        f"unique_ids={rad_df['id'].nunique()}"
    )

    return rad_df.reset_index(drop=True)


def load_deep(modality, audit_dir=MERGE_AUDIT_DIR):
    feat_path = os.path.join(DEEP_ROOT, f"{modality}_medsiglip.csv")
    if not os.path.exists(feat_path):
        raise FileNotFoundError(f"Missing file: {feat_path}")

    deep_df = pd.read_csv(feat_path).copy()

    deep_df["raw_id"] = deep_df["id"].astype(str)
    deep_df["id"] = deep_df["raw_id"].apply(normalize_id)

    audit_normalization_collisions(deep_df, f"{modality}_deep_raw", raw_id_col="raw_id", id_col="id", audit_dir=audit_dir)

    deep_df = audit_and_resolve_duplicates(
        deep_df,
        name=f"{modality}_deep",
        id_col="id",
        raw_id_col="raw_id",
        audit_dir=audit_dir,
    )

    deep_df[f"PRESENT_{modality}_DEEP"] = 1

    rename_map = {}
    for c in deep_df.columns:
        if c not in ["id", "raw_id", f"PRESENT_{modality}_DEEP"]:
            rename_map[c] = f"{modality}_DEEP_{c}"

    deep_df = deep_df.rename(columns=rename_map)
    deep_df = deep_df.drop(columns=["raw_id"])

    print(f"[{modality}_DEEP] rows={len(deep_df)}, unique_ids={deep_df['id'].nunique()}")
    return deep_df.reset_index(drop=True)


def validate_columns(df, cols, context):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{context}: missing columns: {missing}")


def build_all8_table(master_df, audit_dir=MERGE_AUDIT_DIR):
    merged = master_df[["id", "label", "site"]].copy()
    branch_specs = {}

    for modality in IMAGE_MODALITIES:
        rad_df = load_radiomics(modality, audit_dir=audit_dir)
        deep_df = load_deep(modality, audit_dir=audit_dir)

        merged = audited_left_merge(
            merged,
            rad_df,
            left_name="master",
            right_name=f"{modality}_rad",
            key="id",
            audit_dir=audit_dir,
        )
        merged = audited_left_merge(
            merged,
            deep_df,
            left_name=f"master_plus_{modality}_rad",
            right_name=f"{modality}_deep",
            key="id",
            audit_dir=audit_dir,
        )

        rad_cols = [c for c in merged.columns if c.startswith(f"{modality}_RAD_")]
        deep_cols = [c for c in merged.columns if c.startswith(f"{modality}_DEEP_")]

        if len(rad_cols) == 0:
            raise ValueError(f"No radiomics columns found for {modality}")
        if len(deep_cols) == 0:
            raise ValueError(f"No deep columns found for {modality}")

        branch_specs[f"{modality}_RAD"] = {
            "branch_type": "array_numeric",
            "cols": rad_cols,
            "impute": "mean",
            "hidden": BRANCH_CONFIG[f"{modality}_RAD"]["hidden"],
        }
        branch_specs[f"{modality}_DEEP"] = {
            "branch_type": "array_numeric",
            "cols": deep_cols,
            "impute": "mean",
            "hidden": BRANCH_CONFIG[f"{modality}_DEEP"]["hidden"],
        }

    validate_columns(master_df, MRE_NUMERIC_COLS + MRE_CATEGORICAL_COLS, "MRE branch")
    validate_columns(master_df, CLINICAL_NUMERIC_COLS + CLINICAL_CATEGORICAL_COLS, "Clinical branch")

    tabular_df = master_df[[
        "id",
        *sorted(set(MRE_NUMERIC_COLS + MRE_CATEGORICAL_COLS + CLINICAL_NUMERIC_COLS + CLINICAL_CATEGORICAL_COLS))
    ]].copy()

    tabular_df = audit_and_resolve_duplicates(
        tabular_df,
        name="master_tabular",
        id_col="id",
        raw_id_col=None,
        audit_dir=audit_dir,
    )

    tabular_df = add_presence_marker(
        tabular_df,
        "PRESENT_MRE",
        MRE_NUMERIC_COLS + MRE_CATEGORICAL_COLS,
    )
    tabular_df = add_presence_marker(
        tabular_df,
        "PRESENT_CLINICAL",
        CLINICAL_NUMERIC_COLS + CLINICAL_CATEGORICAL_COLS,
    )

    merged = audited_left_merge(
        merged,
        tabular_df,
        left_name="imaging_merged",
        right_name="master_tabular",
        key="id",
        audit_dir=audit_dir,
    )

    branch_specs["MRE"] = {
        "branch_type": "tabular",
        "numeric_cols": MRE_NUMERIC_COLS,
        "categorical_cols": MRE_CATEGORICAL_COLS,
        "hidden": BRANCH_CONFIG["MRE"]["hidden"],
    }
    branch_specs["CLINICAL"] = {
        "branch_type": "tabular",
        "numeric_cols": CLINICAL_NUMERIC_COLS,
        "categorical_cols": CLINICAL_CATEGORICAL_COLS,
        "hidden": BRANCH_CONFIG["CLINICAL"]["hidden"],
    }

    summarize_branch_presence(merged, audit_dir=audit_dir)
    save_csv(merged[["id", "site", "label"]], os.path.join(audit_dir, "final_merged_ids.csv"))

    return merged, branch_specs


# ================= PREPROCESSING =================
def fit_imputer(X_train, strategy="mean"):
    if strategy == "mean":
        stats = np.nanmean(X_train, axis=0)
    elif strategy == "median":
        stats = np.nanmedian(X_train, axis=0)
    else:
        raise ValueError(f"Unknown impute strategy: {strategy}")

    stats = np.where(np.isnan(stats), 0.0, stats)
    return stats.astype(np.float32)


def apply_imputer(X, stats):
    X = X.copy()
    inds = np.where(np.isnan(X))
    X[inds] = stats[inds[1]]
    return X


def fit_numeric_array_branch(df, cols, impute_strategy="mean"):
    X = df[cols].to_numpy(dtype=np.float32)
    X[np.isinf(X)] = np.nan
    stats = fit_imputer(X, strategy=impute_strategy)
    X = apply_imputer(X, stats)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    preprocessor = {
        "kind": "array_numeric",
        "cols": cols,
        "stats": stats,
        "scaler": scaler,
        "impute": impute_strategy,
    }
    return X.astype(np.float32), preprocessor


def transform_numeric_array_branch(df, preprocessor):
    X = df[preprocessor["cols"]].to_numpy(dtype=np.float32)
    X[np.isinf(X)] = np.nan
    X = apply_imputer(X, preprocessor["stats"])
    X = preprocessor["scaler"].transform(X)
    return X.astype(np.float32)


def fit_numeric_tabular(df, numeric_cols):
    if len(numeric_cols) == 0:
        return np.empty((len(df), 0), dtype=np.float32), {
            "numeric_cols": [],
            "stats": np.array([], dtype=np.float32),
            "scaler": None,
        }

    X = df[numeric_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    X[np.isinf(X)] = np.nan

    stats = fit_imputer(X, strategy="median")
    X = apply_imputer(X, stats)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X.astype(np.float32), {
        "numeric_cols": numeric_cols,
        "stats": stats,
        "scaler": scaler,
    }


def transform_numeric_tabular(df, preprocessor):
    if len(preprocessor["numeric_cols"]) == 0:
        return np.empty((len(df), 0), dtype=np.float32)

    X = df[preprocessor["numeric_cols"]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    X[np.isinf(X)] = np.nan
    X = apply_imputer(X, preprocessor["stats"])
    X = preprocessor["scaler"].transform(X)
    return X.astype(np.float32)


def fit_categorical_encoder(df, categorical_cols):
    categories = {}
    encoded_frames = []
    encoded_cols = []

    for col in categorical_cols:
        ser = df[col].fillna("Missing").astype(str)
        cats = sorted(ser.unique().tolist())
        categories[col] = cats

        dummies = pd.get_dummies(pd.Categorical(ser, categories=cats), prefix=col)
        encoded_frames.append(dummies)
        encoded_cols.extend(dummies.columns.tolist())

    if encoded_frames:
        X_cat = pd.concat(encoded_frames, axis=1)
    else:
        X_cat = pd.DataFrame(index=df.index)

    return X_cat.to_numpy(dtype=np.float32), {
        "categorical_cols": categorical_cols,
        "categories": categories,
        "encoded_cols": encoded_cols,
    }


def transform_categorical(df, encoder):
    frames = []

    for col in encoder["categorical_cols"]:
        ser = df[col].fillna("Missing").astype(str)
        cats = encoder["categories"][col]
        dummies = pd.get_dummies(pd.Categorical(ser, categories=cats), prefix=col)
        frames.append(dummies)

    if frames:
        X_cat = pd.concat(frames, axis=1)
        X_cat = X_cat.reindex(columns=encoder["encoded_cols"], fill_value=0.0)
    else:
        X_cat = pd.DataFrame(index=df.index)

    return X_cat.to_numpy(dtype=np.float32)


def fit_tabular_branch(df, numeric_cols, categorical_cols):
    X_num, num_pre = fit_numeric_tabular(df, numeric_cols)
    X_cat, cat_pre = fit_categorical_encoder(df, categorical_cols)
    X = np.hstack([X_num, X_cat]).astype(np.float32)

    preprocessor = {
        "kind": "tabular",
        "numeric": num_pre,
        "categorical": cat_pre,
    }
    return X, preprocessor


def transform_tabular_branch(df, preprocessor):
    X_num = transform_numeric_tabular(df, preprocessor["numeric"])
    X_cat = transform_categorical(df, preprocessor["categorical"])
    return np.hstack([X_num, X_cat]).astype(np.float32)


def fit_preprocessors_and_make_tensors(train_df, branch_specs):
    X_train_dict = {}
    preprocessors = {}
    branch_input_dims = {}

    for branch_name, spec in branch_specs.items():
        if spec["branch_type"] == "array_numeric":
            X_tr, pre = fit_numeric_array_branch(
                train_df,
                cols=spec["cols"],
                impute_strategy=spec.get("impute", "mean"),
            )
        elif spec["branch_type"] == "tabular":
            X_tr, pre = fit_tabular_branch(
                train_df,
                numeric_cols=spec["numeric_cols"],
                categorical_cols=spec["categorical_cols"],
            )
        else:
            raise ValueError(f"Unknown branch_type: {spec['branch_type']}")

        preprocessors[branch_name] = pre
        branch_input_dims[branch_name] = {
            "input_dim": int(X_tr.shape[1]),
            "hidden": int(spec["hidden"]),
        }
        X_train_dict[branch_name] = torch.tensor(X_tr, dtype=torch.float32, device=DEVICE)

    return X_train_dict, preprocessors, branch_input_dims


def transform_df_to_tensors(df, branch_specs, preprocessors):
    X_dict = {}

    for branch_name, spec in branch_specs.items():
        pre = preprocessors[branch_name]

        if spec["branch_type"] == "array_numeric":
            X = transform_numeric_array_branch(df, pre)
        elif spec["branch_type"] == "tabular":
            X = transform_tabular_branch(df, pre)
        else:
            raise ValueError(f"Unknown branch_type: {spec['branch_type']}")

        X_dict[branch_name] = torch.tensor(X, dtype=torch.float32, device=DEVICE)

    return X_dict


def build_presence_dict(df, branch_names):
    presence = {}
    for branch_name in branch_names:
        col = f"PRESENT_{branch_name}"
        if col in df.columns:
            presence[branch_name] = torch.tensor(df[col].fillna(0).to_numpy(dtype=np.float32), dtype=torch.float32, device=DEVICE)
        else:
            presence[branch_name] = torch.ones(len(df), dtype=torch.float32, device=DEVICE)
    return presence


# ================= METRICS =================
def safe_roc_auc(y_true, y_prob):
    try:
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    # More robust for subgroup reporting:
    # if a subgroup has only one class in a fold, BA is not really meaningful.
    if len(np.unique(y_true)) < 2:
        balanced_acc = np.nan
    else:
        balanced_acc = balanced_accuracy_score(y_true, y_pred)

    auc = safe_roc_auc(y_true, y_prob)

    return {
        "balanced_accuracy": balanced_acc,
        "auc": auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def compute_group_metrics(y_true, y_prob, mask, threshold=0.5):
    mask = np.asarray(mask).astype(bool)
    n = int(mask.sum())

    if n == 0:
        return {
            "n_samples": 0,
            "n_pos": 0,
            "n_neg": 0,
            "balanced_accuracy": np.nan,
            "auc": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "tp": 0,
            "tn": 0,
            "fp": 0,
            "fn": 0,
        }

    y_true_sub = np.asarray(y_true)[mask].astype(int)
    y_prob_sub = np.asarray(y_prob)[mask].astype(float)

    out = compute_binary_metrics(y_true_sub, y_prob_sub, threshold=threshold)
    out["n_samples"] = int(len(y_true_sub))
    out["n_pos"] = int((y_true_sub == 1).sum())
    out["n_neg"] = int((y_true_sub == 0).sum())
    return out

def make_sample_weights(y):
    y = np.asarray(y).astype(int)
    n = len(y)
    n0 = max((y == 0).sum(), 1)
    n1 = max((y == 1).sum(), 1)
    w0 = n / (2.0 * n0)
    w1 = n / (2.0 * n1)
    return np.where(y == 0, w0, w1).astype(np.float32)


# ================= MODEL =================
class BranchEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, proj_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class PresenceAwareGate(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim + 1, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, emb, present):
        x = torch.cat([emb, present.unsqueeze(1)], dim=1)
        return self.net(x).squeeze(1)


class HierarchicalGatedFusion(nn.Module):
    def __init__(self, branch_input_dims):
        super().__init__()
        self.branch_names = list(branch_input_dims.keys())
        self.imaging_modalities = ["ADC", "T1", "T2"]
        self.other_modalities = ["MRE", "CLINICAL"]
        self.all_modalities = self.imaging_modalities + self.other_modalities

        self.branch_encoders = nn.ModuleDict()
        self.branch_gates = nn.ModuleDict()
        for branch_name in self.branch_names:
            in_dim = branch_input_dims[branch_name]["input_dim"]
            hidden = branch_input_dims[branch_name]["hidden"]
            self.branch_encoders[branch_name] = BranchEncoder(
                in_dim=in_dim,
                hidden_dim=max(hidden, COMMON_EMBED_DIM),
                proj_dim=COMMON_EMBED_DIM,
                dropout=ENCODER_DROPOUT,
            )
            self.branch_gates[branch_name] = PresenceAwareGate(COMMON_EMBED_DIM)

        self.modality_gates = nn.ModuleDict({
            modality: PresenceAwareGate(COMMON_EMBED_DIM)
            for modality in self.all_modalities
        })
        self.aux_heads = nn.ModuleDict({
            modality: nn.Linear(COMMON_EMBED_DIM, 1)
            for modality in self.all_modalities
        })

        final_in_dim = COMMON_EMBED_DIM * (1 + len(self.all_modalities)) + len(self.all_modalities)
        layers = []
        in_dim = final_in_dim
        for h in FUSION_HIDDEN_DIMS:
            layers.extend([
                nn.Linear(in_dim, h),
                nn.LayerNorm(h),
                nn.ReLU(),
                nn.Dropout(FUSION_DROPOUT),
            ])
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.classifier = nn.Sequential(*layers)

    def _masked_softmax(self, scores, valid_mask):
        masked_scores = scores.masked_fill(valid_mask == 0, -1e9)
        weights = torch.softmax(masked_scores, dim=1)
        weights = weights * valid_mask
        denom = weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        return weights / denom

    def _apply_modality_dropout(self, modality_mask):
        if (not self.training) or MODALITY_DROPOUT <= 0:
            return modality_mask
        drop = (torch.rand_like(modality_mask) > MODALITY_DROPOUT).float()
        kept = modality_mask * drop
        no_keep = kept.sum(dim=1, keepdim=True) == 0
        return torch.where(no_keep, modality_mask, kept)

    def forward(self, x_dict, presence_dict):
        branch_embs = {}
        for branch_name in self.branch_names:
            branch_embs[branch_name] = self.branch_encoders[branch_name](x_dict[branch_name])

        modality_embs = {}
        modality_presence = {}

        for modality in self.imaging_modalities:
            branch_list = [f"{modality}_RAD", f"{modality}_DEEP"]
            emb_stack = torch.stack([branch_embs[b] for b in branch_list], dim=1)
            present_stack = torch.stack([presence_dict[b] for b in branch_list], dim=1)
            score_stack = torch.stack(
                [self.branch_gates[b](branch_embs[b], presence_dict[b]) for b in branch_list],
                dim=1,
            )
            weights = self._masked_softmax(score_stack, present_stack)
            modality_emb = (weights.unsqueeze(-1) * emb_stack).sum(dim=1)
            modality_present = (present_stack.sum(dim=1) > 0).float()
            modality_embs[modality] = modality_emb
            modality_presence[modality] = modality_present

        for modality in self.other_modalities:
            modality_embs[modality] = branch_embs[modality]
            modality_presence[modality] = presence_dict[modality]

        modality_emb_stack = torch.stack([modality_embs[m] for m in self.all_modalities], dim=1)
        modality_present_stack = torch.stack([modality_presence[m] for m in self.all_modalities], dim=1)
        modality_present_stack = self._apply_modality_dropout(modality_present_stack)

        modality_score_stack = torch.stack(
            [self.modality_gates[m](modality_embs[m], modality_present_stack[:, i]) for i, m in enumerate(self.all_modalities)],
            dim=1,
        )
        modality_weights = self._masked_softmax(modality_score_stack, modality_present_stack)
        pooled = (modality_weights.unsqueeze(-1) * modality_emb_stack).sum(dim=1)

        final_feats = [pooled] + [modality_embs[m] for m in self.all_modalities] + [modality_present_stack]
        final_x = torch.cat(final_feats, dim=1)
        main_logit = self.classifier(final_x).squeeze(1)

        aux_logits = {
            modality: self.aux_heads[modality](modality_embs[modality]).squeeze(1)
            for modality in self.all_modalities
        }
        return main_logit, aux_logits, modality_present_stack


# ================= TRAIN / EVAL =================
def predict_probs(model, X_dict, presence_dict):
    model.eval()
    with torch.no_grad():
        logits, _, _ = model(X_dict, presence_dict)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
    return probs


def compute_loss(model, X_dict, presence_dict, y_true_t, w_true_t=None):
    main_logit, aux_logits, modality_present_stack = model(X_dict, presence_dict)
    loss_fn = nn.BCEWithLogitsLoss(reduction="none")

    main_loss_vec = loss_fn(main_logit, y_true_t)
    if w_true_t is not None:
        main_loss = (main_loss_vec * w_true_t).mean()
    else:
        main_loss = main_loss_vec.mean()

    aux_losses = []
    for idx, modality in enumerate(ALL_MODALITIES):
        mask = modality_present_stack[:, idx]
        if float(mask.sum().item()) <= 0:
            continue
        aux_loss_vec = loss_fn(aux_logits[modality], y_true_t)
        if w_true_t is not None:
            aux_loss_vec = aux_loss_vec * w_true_t
        aux_loss = (aux_loss_vec * mask).sum() / mask.sum().clamp_min(1.0)
        aux_losses.append(aux_loss)

    if aux_losses:
        aux_loss = torch.stack(aux_losses).mean()
        total_loss = main_loss + AUX_LOSS_WEIGHT * aux_loss
    else:
        total_loss = main_loss

    return total_loss, main_logit


def train_model(
    X_train_dict,
    train_presence_dict,
    y_train,
    branch_input_dims,
    X_val_dict=None,
    val_presence_dict=None,
    y_val=None,
    early_stopping_patience=None,
    min_epochs=1,
):
    model = HierarchicalGatedFusion(branch_input_dims=branch_input_dims).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    y_train_t = torch.tensor(y_train.astype(np.float32), dtype=torch.float32, device=DEVICE)
    w_train_t = torch.tensor(make_sample_weights(y_train), dtype=torch.float32, device=DEVICE)

    y_val_t = None
    if y_val is not None:
        y_val_t = torch.tensor(y_val.astype(np.float32), dtype=torch.float32, device=DEVICE)

    use_early_stopping = (
        X_val_dict is not None
        and val_presence_dict is not None
        and y_val is not None
        and early_stopping_patience is not None
    )

    best_state_dict = None
    best_val_loss = np.inf
    epochs_without_improvement = 0

    n_train = len(y_train)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)

        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            batch_x_dict = {k: v[idx] for k, v in X_train_dict.items()}
            batch_presence_dict = {k: v[idx] for k, v in train_presence_dict.items()}
            batch_y = y_train_t[idx]
            batch_w = w_train_t[idx]

            loss, _ = compute_loss(model, batch_x_dict, batch_presence_dict, batch_y, batch_w)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        if use_early_stopping:
            model.eval()
            with torch.no_grad():
                val_loss, _ = compute_loss(model, X_val_dict, val_presence_dict, y_val_t, None)
                val_loss_value = float(val_loss.item())

            if val_loss_value < best_val_loss:
                best_val_loss = val_loss_value
                best_state_dict = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epoch >= min_epochs and epochs_without_improvement >= early_stopping_patience:
                break

    if use_early_stopping and best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    return model


def make_inner_train_val_split(train_df, val_frac=0.15):
    stratify_values = choose_train_test_strata(
        train_df,
        context="cv_inner_early_stop_split",
    )

    try:
        tr_df, val_df = train_test_split(
            train_df,
            test_size=val_frac,
            random_state=SEED,
            stratify=stratify_values,
        )
    except ValueError as e:
        print(f"[cv_inner_early_stop_split] WARNING: stratified split failed: {e}")
        tr_df, val_df = train_test_split(
            train_df,
            test_size=val_frac,
            random_state=SEED,
            shuffle=True,
        )

    return tr_df.reset_index(drop=True), val_df.reset_index(drop=True)


def run_dev_cv(df, branch_specs):
    if "AGE_years" not in df.columns:
        raise ValueError("run_dev_cv requires AGE_years in df for adult/child subgroup metrics.")

    y = df["label"].values.astype(int)
    cv_strata, cv_stratification_mode = choose_cv_strata(
        df,
        n_splits=N_SPLITS,
        context="dev_cv_outer",
    )
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_rows = []
    fold_balance_rows = []

    for fold_idx, (tr, val) in enumerate(skf.split(np.zeros(len(df)), cv_strata), start=1):
        outer_train_df = df.iloc[tr].reset_index(drop=True)
        outer_val_df = df.iloc[val].reset_index(drop=True)

        # Save a transparent fold-balance audit. This makes it easy to check
        # that each fold is balanced by site and fibrosis class.
        for split_name, split_df in [
            ("outer_train", outer_train_df),
            ("outer_val", outer_val_df),
        ]:
            tmp = (
                split_df.groupby(["site", "label"], dropna=False)
                .size()
                .reset_index(name="n")
            )
            tmp["fold"] = fold_idx
            tmp["split"] = split_name
            fold_balance_rows.append(tmp)

        inner_train_df, inner_stop_df = make_inner_train_val_split(
            outer_train_df,
            val_frac=CV_INNER_VAL_FRAC,
        )

        X_train_dict, preprocessors, branch_input_dims = fit_preprocessors_and_make_tensors(
            inner_train_df,
            branch_specs,
        )
        train_presence_dict = build_presence_dict(inner_train_df, branch_specs.keys())

        X_stop_dict = transform_df_to_tensors(inner_stop_df, branch_specs, preprocessors)
        stop_presence_dict = build_presence_dict(inner_stop_df, branch_specs.keys())

        X_val_dict = transform_df_to_tensors(outer_val_df, branch_specs, preprocessors)
        val_presence_dict = build_presence_dict(outer_val_df, branch_specs.keys())

        y_train = inner_train_df["label"].values.astype(np.float32)
        y_stop = inner_stop_df["label"].values.astype(int)
        y_val = outer_val_df["label"].values.astype(int)

        model = train_model(
            X_train_dict=X_train_dict,
            train_presence_dict=train_presence_dict,
            y_train=y_train,
            branch_input_dims=branch_input_dims,
            X_val_dict=X_stop_dict,
            val_presence_dict=stop_presence_dict,
            y_val=y_stop,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            min_epochs=MIN_EPOCHS,
        )

        val_probs = predict_probs(model, X_val_dict, val_presence_dict)
        overall_metrics = compute_binary_metrics(y_val, val_probs, threshold=FIXED_THRESHOLD)

        age_years_val = pd.to_numeric(outer_val_df["AGE_years"], errors="coerce").to_numpy()
        child_mask = age_years_val <= child_cutoff
        adult_mask = age_years_val > child_cutoff

        child_metrics = compute_group_metrics(
            y_true=y_val,
            y_prob=val_probs,
            mask=child_mask,
            threshold=FIXED_THRESHOLD,
        )
        adult_metrics = compute_group_metrics(
            y_true=y_val,
            y_prob=val_probs,
            mask=adult_mask,
            threshold=FIXED_THRESHOLD,
        )

        row = {
            "fold": fold_idx,
            "cv_stratification": cv_stratification_mode,
            "n_inner_train": len(inner_train_df),
            "n_inner_stop": len(inner_stop_df),
            "n_outer_val": len(outer_val_df),
            "threshold": FIXED_THRESHOLD,

            # overall
            "balanced_accuracy": overall_metrics["balanced_accuracy"],
            "auc": overall_metrics["auc"],
            "sensitivity": overall_metrics["sensitivity"],
            "specificity": overall_metrics["specificity"],
            "tp": overall_metrics["tp"],
            "tn": overall_metrics["tn"],
            "fp": overall_metrics["fp"],
            "fn": overall_metrics["fn"],

            # children
            "child_n": child_metrics["n_samples"],
            "child_n_pos": child_metrics["n_pos"],
            "child_n_neg": child_metrics["n_neg"],
            "child_balanced_accuracy": child_metrics["balanced_accuracy"],
            "child_auc": child_metrics["auc"],
            "child_sensitivity": child_metrics["sensitivity"],
            "child_specificity": child_metrics["specificity"],
            "child_tp": child_metrics["tp"],
            "child_tn": child_metrics["tn"],
            "child_fp": child_metrics["fp"],
            "child_fn": child_metrics["fn"],

            # adults
            "adult_n": adult_metrics["n_samples"],
            "adult_n_pos": adult_metrics["n_pos"],
            "adult_n_neg": adult_metrics["n_neg"],
            "adult_balanced_accuracy": adult_metrics["balanced_accuracy"],
            "adult_auc": adult_metrics["auc"],
            "adult_sensitivity": adult_metrics["sensitivity"],
            "adult_specificity": adult_metrics["specificity"],
            "adult_tp": adult_metrics["tp"],
            "adult_tn": adult_metrics["tn"],
            "adult_fp": adult_metrics["fp"],
            "adult_fn": adult_metrics["fn"],
        }
        row = add_fold_site_label_counts(row, fold_idx, "outer_val", outer_val_df)
        fold_rows.append(row)

        site_counts_msg = (
            outer_val_df.groupby(["site", "label"])
            .size()
            .reset_index(name="n")
            .to_dict("records")
        )
        print(
            f"fold {fold_idx}: "
            f"overall BA={overall_metrics['balanced_accuracy']:.4f}, "
            f"child BA={child_metrics['balanced_accuracy']:.4f} (n={child_metrics['n_samples']}), "
            f"adult BA={adult_metrics['balanced_accuracy']:.4f} (n={adult_metrics['n_samples']}), "
            f"outer-val site/label={site_counts_msg}"
        )

    fold_df = pd.DataFrame(fold_rows)

    if fold_balance_rows:
        fold_balance_df = pd.concat(fold_balance_rows, ignore_index=True)
        save_csv(fold_balance_df, os.path.join(OUTPUT_DIR, "dev_cv_fold_site_label_balance.csv"))

    summary = {
        "model": "gated_hierarchical_multimodal_fusion",
        "cv_stratification": cv_stratification_mode,

        "balanced_accuracy_mean": mean_std(fold_df["balanced_accuracy"])[0],
        "balanced_accuracy_std": mean_std(fold_df["balanced_accuracy"])[1],
        "auc_mean": mean_std(fold_df["auc"])[0],
        "auc_std": mean_std(fold_df["auc"])[1],
        "sensitivity_mean": mean_std(fold_df["sensitivity"])[0],
        "sensitivity_std": mean_std(fold_df["sensitivity"])[1],
        "specificity_mean": mean_std(fold_df["specificity"])[0],
        "specificity_std": mean_std(fold_df["specificity"])[1],

        "child_balanced_accuracy_mean": mean_std(fold_df["child_balanced_accuracy"])[0],
        "child_balanced_accuracy_std": mean_std(fold_df["child_balanced_accuracy"])[1],
        "child_auc_mean": mean_std(fold_df["child_auc"])[0],
        "child_auc_std": mean_std(fold_df["child_auc"])[1],
        "child_sensitivity_mean": mean_std(fold_df["child_sensitivity"])[0],
        "child_sensitivity_std": mean_std(fold_df["child_sensitivity"])[1],
        "child_specificity_mean": mean_std(fold_df["child_specificity"])[0],
        "child_specificity_std": mean_std(fold_df["child_specificity"])[1],

        "adult_balanced_accuracy_mean": mean_std(fold_df["adult_balanced_accuracy"])[0],
        "adult_balanced_accuracy_std": mean_std(fold_df["adult_balanced_accuracy"])[1],
        "adult_auc_mean": mean_std(fold_df["adult_auc"])[0],
        "adult_auc_std": mean_std(fold_df["adult_auc"])[1],
        "adult_sensitivity_mean": mean_std(fold_df["adult_sensitivity"])[0],
        "adult_sensitivity_std": mean_std(fold_df["adult_sensitivity"])[1],
        "adult_specificity_mean": mean_std(fold_df["adult_specificity"])[0],
        "adult_specificity_std": mean_std(fold_df["adult_specificity"])[1],
    }

    return summary, fold_df

def make_final_retrain_split(dev_df, val_frac=0.20):
    stratify_values = choose_train_test_strata(
        dev_df,
        context="final_retrain_internal_val_split",
    )

    try:
        train_df, val_df = train_test_split(
            dev_df,
            test_size=val_frac,
            random_state=SEED,
            stratify=stratify_values,
        )
    except ValueError as e:
        print(f"[final_retrain_internal_val_split] WARNING: stratified split failed: {e}")
        train_df, val_df = train_test_split(
            dev_df,
            test_size=val_frac,
            random_state=SEED,
            shuffle=True,
        )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def retrain_on_dev_and_test_external(dev_df, external_df_dict, branch_specs):
    retrain_train_df, retrain_val_df = make_final_retrain_split(dev_df, FINAL_RETRAIN_VAL_FRAC)

    X_train_dict, preprocessors, branch_input_dims = fit_preprocessors_and_make_tensors(
        retrain_train_df,
        branch_specs,
    )
    train_presence_dict = build_presence_dict(retrain_train_df, branch_specs.keys())

    X_val_dict = transform_df_to_tensors(retrain_val_df, branch_specs, preprocessors)
    val_presence_dict = build_presence_dict(retrain_val_df, branch_specs.keys())

    y_train = retrain_train_df["label"].values.astype(np.float32)
    y_val = retrain_val_df["label"].values.astype(int)

    model = train_model(
        X_train_dict=X_train_dict,
        train_presence_dict=train_presence_dict,
        y_train=y_train,
        branch_input_dims=branch_input_dims,
        X_val_dict=X_val_dict,
        val_presence_dict=val_presence_dict,
        y_val=y_val,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        min_epochs=MIN_EPOCHS,
    )

    val_probs = predict_probs(model, X_val_dict, val_presence_dict)
    internal_metrics = compute_binary_metrics(y_val, val_probs, threshold=FIXED_THRESHOLD)

    internal_row = {
        "split": "DEV_internal_val",
        "balanced_accuracy": internal_metrics["balanced_accuracy"],
        "auc": internal_metrics["auc"],
        "sensitivity": internal_metrics["sensitivity"],
        "specificity": internal_metrics["specificity"],
        "tp": internal_metrics["tp"],
        "tn": internal_metrics["tn"],
        "fp": internal_metrics["fp"],
        "fn": internal_metrics["fn"],
    }

    external_rows = []
    pred_rows = []

    for site_name, ext_df in external_df_dict.items():
        if ext_df.empty:
            continue

        X_ext_dict = transform_df_to_tensors(ext_df, branch_specs, preprocessors)
        ext_presence_dict = build_presence_dict(ext_df, branch_specs.keys())
        y_ext = ext_df["label"].values.astype(int)
        ext_probs = predict_probs(model, X_ext_dict, ext_presence_dict)
        metrics = compute_binary_metrics(y_ext, ext_probs, threshold=FIXED_THRESHOLD)

        external_rows.append({
            "site": site_name,
            "n_samples": len(ext_df),
            "balanced_accuracy": metrics["balanced_accuracy"],
            "auc": metrics["auc"],
            "sensitivity": metrics["sensitivity"],
            "specificity": metrics["specificity"],
            "tp": metrics["tp"],
            "tn": metrics["tn"],
            "fp": metrics["fp"],
            "fn": metrics["fn"],
        })

        pred_df = ext_df[["id", "site", "label"]].copy()
        pred_df["prob"] = ext_probs
        pred_df["pred"] = (ext_probs >= FIXED_THRESHOLD).astype(int)
        pred_rows.append(pred_df)

    internal_df = pd.DataFrame([internal_row])
    external_df = pd.DataFrame(external_rows)
    external_pred_df = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()

    return internal_df, external_df, external_pred_df


def print_dev_cv_results(summary):
    print("DEV CV RESULTS")
    print("Model: gated_hierarchical_multimodal_fusion")
    print(f"Common embed dim: {COMMON_EMBED_DIM}")
    print(f"Fusion head: {FUSION_HIDDEN_DIMS}")
    print(f"Fixed threshold: {FIXED_THRESHOLD:.4f}")
    print(f"LR={LR}, WD={WEIGHT_DECAY}, Batch={BATCH_SIZE}")

    print(f"Overall BA : {summary['balanced_accuracy_mean']:.4f} ± {summary['balanced_accuracy_std']:.4f}")
    print(f"Overall Sen: {summary['sensitivity_mean']:.4f} ± {summary['sensitivity_std']:.4f}")
    print(f"Overall Spe: {summary['specificity_mean']:.4f} ± {summary['specificity_std']:.4f}")
    print(f"Overall AUC: {summary['auc_mean']:.4f} ± {summary['auc_std']:.4f}")
    print()

    print(f"Child   BA : {summary['child_balanced_accuracy_mean']:.4f} ± {summary['child_balanced_accuracy_std']:.4f}")
    print(f"Child   Sen: {summary['child_sensitivity_mean']:.4f} ± {summary['child_sensitivity_std']:.4f}")
    print(f"Child   Spe: {summary['child_specificity_mean']:.4f} ± {summary['child_specificity_std']:.4f}")
    print(f"Child   AUC: {summary['child_auc_mean']:.4f} ± {summary['child_auc_std']:.4f}")
    print()

    print(f"Adult   BA : {summary['adult_balanced_accuracy_mean']:.4f} ± {summary['adult_balanced_accuracy_std']:.4f}")
    print(f"Adult   Sen: {summary['adult_sensitivity_mean']:.4f} ± {summary['adult_sensitivity_std']:.4f}")
    print(f"Adult   Spe: {summary['adult_specificity_mean']:.4f} ± {summary['adult_specificity_std']:.4f}")
    print(f"Adult   AUC: {summary['adult_auc_mean']:.4f} ± {summary['adult_auc_std']:.4f}")
    print()


def print_external_results(internal_df, external_df):
    internal_row = internal_df.iloc[0].to_dict()
    print("EXTERNAL TEST RESULTS")
    for _, row in external_df.iterrows():
        print(f"gated_hierarchical_multimodal_fusion | External {row['site']}")
        print(f"n={int(row['n_samples'])}")
        print(
            f"Fixed threshold: {FIXED_THRESHOLD:.4f} "
            f"(retrain-val BA={internal_row['balanced_accuracy']:.4f})"
        )
        print(f"BA : {row['balanced_accuracy']:.4f}")
        print(f"Sen: {row['sensitivity']:.4f}")
        print(f"Spe: {row['specificity']:.4f}")
        print(f"AUC: {row['auc']:.4f}")
        print(
            f"TP={int(row['tp'])}, TN={int(row['tn'])}, "
            f"FP={int(row['fp'])}, FN={int(row['fn'])}"
        )
        print()


# ================= MAIN =================
def main():
    set_seed(SEED)
    print(f"Using device: {DEVICE}\n")
    print(f"Merge audit dir: {MERGE_AUDIT_DIR}\n")

    master_df = load_master_table(audit_dir=MERGE_AUDIT_DIR)
    merged_df, branch_specs = build_all8_table(master_df, audit_dir=MERGE_AUDIT_DIR)

    dev_df = merged_df[merged_df["site"].isin(DEV_SITES)].copy().reset_index(drop=True)
    external_df_dict = {
        site: merged_df[merged_df["site"] == site].copy().reset_index(drop=True)
        for site in EXTERNAL_SITES
    }

    summary, fold_df = run_dev_cv(
        df=dev_df,
        branch_specs=branch_specs,
    )
    print_dev_cv_results(summary)

    internal_df, external_results_df, external_pred_df = retrain_on_dev_and_test_external(
        dev_df=dev_df,
        external_df_dict=external_df_dict,
        branch_specs=branch_specs,
    )
    print_external_results(internal_df, external_results_df)

    pd.DataFrame([summary]).to_csv(os.path.join(OUTPUT_DIR, "dev_cv_summary.csv"), index=False)
    fold_df.to_csv(os.path.join(OUTPUT_DIR, "dev_cv_fold_results.csv"), index=False)
    internal_df.to_csv(os.path.join(OUTPUT_DIR, "dev_internal_val_results.csv"), index=False)
    external_results_df.to_csv(os.path.join(OUTPUT_DIR, "external_test_results.csv"), index=False)
    external_pred_df.to_csv(os.path.join(OUTPUT_DIR, "external_test_predictions.csv"), index=False)

    with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w") as f:
        json.dump(
            {
                "model": "gated_hierarchical_multimodal_fusion",
                "common_embed_dim": COMMON_EMBED_DIM,
                "fusion_hidden_dims": FUSION_HIDDEN_DIMS,
                "encoder_dropout": ENCODER_DROPOUT,
                "fusion_dropout": FUSION_DROPOUT,
                "modality_dropout": MODALITY_DROPOUT,
                "aux_loss_weight": AUX_LOSS_WEIGHT,
                "lr": LR,
                "weight_decay": WEIGHT_DECAY,
                "batch_size": BATCH_SIZE,
                "fixed_threshold": FIXED_THRESHOLD,
                "cv_stratification": "site_label_joint_when_possible",
                "dev_sites": DEV_SITES,
                "external_sites": EXTERNAL_SITES,
                "branch_config": BRANCH_CONFIG,
                "merge_audit_dir": MERGE_AUDIT_DIR,
            },
            f,
            indent=2,
        )

if __name__ == "__main__":
    main()

Using device: cuda

Merge audit dir: ../Results/gated_hierarchical_multimodal_fusion/merge_audit

[master_raw] normalization collisions: 0
[master_before_site_filter] site mismatches vs ID pattern: 0
[master_filtered] rows=520, unique_ids=520, duplicate_ids=0
[master_filtered] final rows=520, unique_ids=520, site_counts={'CCHMC': 236, 'MICH': 46, 'NYU': 80, 'WISC': 158}
[ADC_liver_raw] normalization collisions: 0
[ADC_spleen_raw] normalization collisions: 0
[ADC_liver] rows=526, unique_ids=526, duplicate_ids=0
[ADC_spleen] rows=526, unique_ids=526, duplicate_ids=0
left_name right_name  left_rows  right_rows  left_unique_ids  right_unique_ids  matched_ids  left_only_ids  right_only_ids
ADC_liver ADC_spleen        526         526              526               526          526              0               0
[ADC_RAD] rows after liver/spleen inner merge=526, unique_ids=526
[ADC_deep_raw] normalization collisions: 0
[ADC_deep] rows=273, unique_ids=273, duplicate_ids=0
[ADC_DEEP] rows=273, 